In [1]:
import sys; sys.path.append('../..')
from abstraction.llm import *
from abstraction.utils import *
from abstraction.scoring import *
from collections import defaultdict
from tqdm import tqdm

In [2]:
LLM_STASH = get_llm_stash()

In [3]:

def load_results(stash, model=DEFAULT_MODEL):
    id2data = defaultdict(list)
    for k,v in tqdm(stash.items(), total=len(stash)):
        if k['model'] != model:
            continue

        v = parse_json_str(v)
        if v:
            slice_txts = k['user_prompt'].split('#########\n\n')[1:]
            if len(slice_txts) != len(v):
                print(f"Warning: {k['text_id']} has {len(slice_txts)} slices but {len(v)} results")
                continue
            for txt,v_i in zip(slice_txts,v):
                v_i['slice_txt'] = txt
                v_i['concabs'] = score_psg(txt)
            id2data[k['text_id']].extend(v)
    return id2data
            

In [4]:
results = load_results(LLM_STASH)
# novel_results = results['Richardson.Pamela']
# novel_results.sort(key=lambda x: x['id'])
# novel_results[0]

 86%|████████▌ | 161/188 [00:01<00:00, 135.95it/s]

Error parsing JSON: Expecting value: line 107 column 1 (char 3194)
Error parsing JSON: Expecting value: line 63 column 1 (char 2129)
Error parsing JSON: Expecting ',' delimiter: line 173 column 253 (char 5051)


100%|██████████| 188/188 [00:01<00:00, 113.91it/s]


In [5]:
# results

In [6]:
def compile_dataframe(results, **metadata):
    """
    Compile slice-level JSON results into a flat dataframe.
    
    Returns two dataframes:
    - slices_df: one row per slice (slice-level annotations)
    - chars_df: one row per character-description per slice
      (with slice-level columns duplicated)
    """
    
    slice_rows = []
    char_rows = []
    
    for s in results:
        sid = s['id']
        slice_num = int(sid.split('_')[1])
        
        # Slice-level row
        slice_row = {
            **metadata,
            'slice_id': sid,
            'slice_num': slice_num,
            'slice_concabs': s.get('concabs', None),
            'summary': s.get('summary', ''),
            'narrative_mode': s.get('narrative_mode', ''),
            'concrete_abstract': s.get('concrete_abstract', None),
            'social_space': s.get('social_space', ''),
            'n_char_descriptions': len(s.get('character_descriptions', [])),
            'key_abstractions': s.get('key_abstractions', []),
            'key_concretions': s.get('key_concretions', []),
            'n_abstractions': len(s.get('key_abstractions', [])),
            'n_concretions': len(s.get('key_concretions', [])),
            'slice_txt': s.get('slice_txt', ''),
        }
        slice_rows.append(slice_row)
        
        # Character-level rows
        for desc in s.get('character_descriptions', []):
            # Normalize mode to always be a list
            mode = desc.get('mode', [])
            if isinstance(mode, str):
                mode = [mode]
            
            char_row = {
                # Slice-level info (duplicated)
                **metadata,
                'slice_id': sid,
                'slice_num': slice_num,
                'slice_concabs': s.get('concabs', None),
                'narrative_mode': s.get('narrative_mode', ''),
                'concrete_abstract': s.get('concrete_abstract', None),
                'social_space': s.get('social_space', ''),
                
                # Character-level info
                'character': desc.get('name', ''),
                'gender': desc.get('gender', 'unknown'),
                'class': desc.get('class', 'unknown'),
                'described_by': desc.get('described_by', ''),
                'passage': desc.get('passage', ''),
                'descriptors': desc.get('descriptors', []),
                'mode': mode,
                
                # Flattened mode booleans for easy filtering
                'mode_physical': 'physical' in mode,
                'mode_social': 'social' in mode,
                'mode_moral': 'moral' in mode,
                'mode_manner': 'manner' in mode,
                'mode_material': 'material' in mode,
                'mode_relational': 'relational' in mode,
                'mode_behavioral': 'behavioral' in mode,
                
                # Described-by categories
                'by_narrator': desc.get('described_by', '').startswith('narrator'),
                'by_other_char': desc.get('described_by', '').startswith('other_character'),
                'by_collective': desc.get('described_by', '').startswith('collective'),
            }
            char_rows.append(char_row)
    
    slices_df = pd.DataFrame(slice_rows)
    chars_df = pd.DataFrame(char_rows)
    
    return co_presence_ratio(slices_df), first_appearances(chars_df)


def first_appearances(chars_df):
    """
    For each character in each novel, find their first description.
    Everything after is a redescription.
    """
    chars_df = chars_df.copy()
    chars_df = chars_df.sort_values(['novel', 'character', 'slice_num'])
    chars_df['is_introduction'] = ~chars_df.duplicated(
        subset=['novel', 'character'], keep='first'
    )
    return chars_df


def co_presence_ratio(slices_df):
    """
    Compute abstract/concrete co-presence ratio per slice.
    High ratio = both registers loaded (allegorical).
    Low ratio = one register dominates.
    """
    df = slices_df.copy()
    df['co_presence'] = df.apply(
        lambda r: (
            min(r['n_abstractions'], r['n_concretions']) /
            max(r['n_abstractions'], r['n_concretions'])
            if max(r['n_abstractions'], r['n_concretions']) > 0
            else 0
        ),
        axis=1
    )
    return df


In [7]:
# Example: load and compile multiple novels

all_slices = []
all_chars = []

for novel,novel_results in results.items():
    slices_df, chars_df = compile_dataframe(
        novel_results, novel=novel
    )
    all_slices.append(slices_df)
    all_chars.append(chars_df)

slices = pd.concat(all_slices, ignore_index=True)
chars = pd.concat(all_chars, ignore_index=True)

# Add co-presence ratio
slices = co_presence_ratio(slices)

# Mark first appearances vs redescriptions
chars = first_appearances(chars)


In [8]:
pamela1_slices = slices[slices.novel == 'Richardson.Pamela']
pamela2_slices = slices[slices.novel == 'Richardson.Pamela.Vol2']
# pamela2_slices['slice_num'] = pamela2_slices['slice_num'] + len(pamela1_slices)
pamela2_slices['slice_num'] = pamela2_slices['slice_num'] + pamela1_slices.slice_num.max()
pamela_slices = pd.concat([pamela1_slices, pamela2_slices])
pamela_slices[['slice_num','slice_txt']]

,slice_num,slice_txt
628,30,"\n\nSo, as I was saying, unknown to any body, ..."
629,31,"\n\nSo, as I was saying, I have provided a new..."
630,32,"\n\nNay, now, Pamela, said she, thou carriest ..."
631,33,"Jervis,\nare we to lose Mrs. Pamela? as they ..."
632,34,"Thank you, Mr. Jonathan, said I; but as you v..."
...,...,...
395,789,\n\nThe task I have undertaken of dedicating a...
396,790,"\n\nYou see what a sweet girl Miss is, and you..."
397,791,"\nAll, in short, was done with cheerful ease a..."
398,792,\n\nFor having happily put a stop to that affa...


In [9]:
pamela_anno = {
  "0": "Novel opens; lady dies",
  "10": "First assault (summer-house)",
  "18": "Second assault; Pamela faints",
  "21": "Dismissed; told to leave",
  "29": "Prepares humble clothes",
  "39": "Disguise scene; mistaken identity",
  "45": "Closet assault; Mrs Jervis defends",
  "59": "Sorts belongings; master hidden",
  "65": "Master confesses love; reads letters",
  "67": "Sham marriage proposed",
  "70": "Abduction by chariot",
  "76": "Journal begins; captivity",
  "85": "Arrives Lincolnshire prison",
  "90": "Mrs Jewkes described",
  "96": "Secret correspondence with Williams",
  "136": "Escape plan devised",
  "139": "Escape fails; falls from wall",
  "140": "Suicide temptation at pond",
  "149": "Mr B arrives at Lincolnshire",
  "153": "Mistress articles proposed",
  "163": "Bed assault in disguise",
  "170": "Promise not to force her",
  "176": "Pond confession of love",
  "200": "Sent away again",
  "207": "Recalled; marriage proposal",
  "209": "Pamela returns",
  "231": "Wedding planned",
  "288": "WEDDING CEREMONY",
  "296": "Money and gifts distributed",
  "318": "Lady Davers arrives",
  "334": "Escapes Lady Davers",
  "350": "Bedroom confrontation",
  "368": "Reconciliation with Lady Davers",
  "387": "Returns to Bedfordshire as mistress",
  "404": "Miss Goodwin revealed as daughter",
  "112": "Bold letter on honour",
  "154": "Rejects mistress articles",
  "256": "Williams reconciliation",
  "370": "Forgives Mrs Jewkes/Worden",
  "374": "Mr B lectures on marriage",
  # "422": "Narrator's moral commentary"
}


pamela_anno = {
    "0": "Novel opens",
    "10": "First assault",
    "45": "Closet assault",
    "70": "Abduction",
    "85": "Arrives Lincolnshire prison",
    # "90": "Mrs Jewkes described",
    "112": "Bold letter on honour",
    # "139": "Escape fails",
    "140": "Suicide temptation",
    "154": "Rejects mistress articles",
    "163": "Bed assault in disguise",
    "176": "Pond confession of love",
    "207": "Recalled; marriage proposal",
    "288": "Wedding ceremony",
    # "288": "WEDDING CEREMONY",
    # "296": "Money and gifts distributed",
    "334": "Escapes Lady Davers",
    "368": "Reconciliation with Lady Davers",
    "370": "Forgives Mrs Jewkes",
    "374": "Mr B lectures on marriage",
    "387": "Returns as mistress",
    "404": "Miss Goodwin revealed",
    # "422": "Narrator's moral commentary",
}

pamela2_anno = {
    # "415": "Vol 2 opens; married life",
    # "458": "Pamela as almoner; Mrs Jervis's debts",
    # "502": "Lady Davers reads assault account aloud",
    "504": "Mr B's confession begins",
    "551": "Sir Jacob Swynford arrives",
    # "559": "Pamela revealed as Mrs B",
    # "578": "Discovers Mr H with Polly",
    "615": "Breastfeeding dispute",
    # "641": "Masquerade as Quaker",
    "650": "Son born",
    # "663": "Countess affair discovered",
    "681": "Pamela's 'trial' of Mr B",
    # "689": "Reconciliation; Mr B reformed",
    "738": "Begins Locke critique",
    # "780": "European tour announced",
    # "800": "Lectures young ladies on love",
    # "824": "Editor's conclusion",
}
pamela2_anno = {int(k) - len(pamela1_slices):v for k,v in pamela2_anno.items()}
pamela2_anno = {k + pamela1_slices.slice_num.max():v for k,v in pamela2_anno.items()}
pamela_anno = {int(k):v for k,v in {**pamela_anno, **pamela2_anno}.items()}

In [10]:
pamela1_slices[['slice_num','slice_txt']].to_csv('/Volumes/diderot/DH/data/data_abslithist/psgs/pamela-summaries.csv', index=False)

In [11]:
# pamela_slices.sort_values('slice_concabs', ascending=False)

In [12]:
# !open /Volumes/diderot/DH/data/data_abslithist/psgs/

In [13]:
# print(pamela_slices.loc[pamela_slices.slice_num == 163, 'slice_txt'].iloc[0])

In [14]:
def clean_slice_text(slice_text):
    slice_text = slice_text.strip().split('---------')[0]
    slice_text = slice_text.replace('\n\n', '||').replace('\n', ' ').replace('||', '\n\n')
    return slice_text

with open('/Volumes/diderot/DH/data/data_abslithist/psgs/pamela2-short.txt', 'r') as f:
    slice_text1 = f.read()

passage1 = {'title': pamela_anno[45], 'text': clean_slice_text(slice_text1)}
passage1

{'title': 'Closet assault',
 'text': "I pulled off my stays, and my stockings, and all my clothes to an under-petticoat; and then hearing a rustling again in the closet, I said, Heaven protect us! but before I say my prayers, I must look into this closet. And so was going to it slip-shod, when, O dreadful! out rushed my master in a rich silk and silver morning gown.\n\nI screamed, and ran to the bed, and Mrs. Jervis screamed too; and he said, I'll do you no harm, if you forbear this noise; but otherwise take what follows.\n\nInstantly he came to the bed (for I had crept into it, to Mrs. Jervis, with my coat on, and my shoes); and taking me in his arms, said, Mrs. Jervis, rise, and just step up stairs to keep the maids from coming down at this noise: I'll do no harm to this rebel. [...]\n\nI found his hand in my bosom; and when my fright let me know it, I was ready to die; and I sighed and screamed, and fainted away. And still he had his arms about my neck; and Mrs. Jervis was about my 

In [15]:
slice_text2 = pamela_slices.loc[pamela_slices.slice_num == 112, 'slice_txt'].iloc[0]

with open('/Volumes/diderot/DH/data/data_abslithist/psgs/pamela3-short.txt', 'r') as f:
    slice_text2 = f.read()

passage2 = {'title': pamela_anno[112], 'text': clean_slice_text(slice_text2)}
passage2

{'title': 'Bold letter on honour',
 'text': "'Whatever rashness you may impute to me, I cannot help it; but I wish I may not be forced upon any, that otherwise would never enter into my thoughts. Forgive me, sir, my plainness; I should be loath to behave to my master unbecomingly; but I must needs say, sir, my innocence is so dear to me, that all other considerations are, and, I hope, shall ever be, treated by me as niceties, that ought, for that, to be dispensed with. If you mean honourably, why, sir, should you not let me know it plainly? Why is it necessary to imprison me, to convince me of it? And why must I be close watched, and attended, hindered from stirring out, from speaking to any body, from going so much as to church to pray for you, who have been, till of late, so generous a benefactor to me? Why, sir, I humbly ask, why all this, if you mean honourably?--It is not for me to expostulate so freely, but in a case so near to me, with you, sir, so greatly my superior. Pardon me

In [16]:
score_psg(slice_text2)

-0.781893735772331

In [17]:
score_psg(slice_text1)

0.1974377057681953

In [18]:
from abstraction.passages import save_comparison_image

save_comparison_image([passage1, passage2], '../figures/pamela_45_112.png')

'../figures/pamela_45_112.png'

In [19]:
# get_allnorms().loc['servant']